# sweep-config-dict — ex1: build a wandb sweep config: bayes + metric + parameters

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sweep-config-dict`. Running the final beacon cell reports progress against the `Config: wandb sweep config dict` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: wandb sweep config dict` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sweep-config-dict`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sweep-config-dict"
DD_SUBTOPIC = "Config: wandb sweep config dict"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: wandb sweep config dict — quick refresher

A wandb sweep is configured by a Python dict (or equivalent YAML) with a fixed top-level schema:

```python
sweep_config = {
    'method': 'bayes',                 # 'grid' | 'random' | 'bayes'
    'metric': {                         # what bayes optimizes
        'name': 'val/loss',
        'goal': 'minimize',
    },
    'parameters': {
        'lr':         {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-1},
        'batch_size': {'values': [16, 32, 64]},
        'optimizer':  {'value': 'adam'},
    },
}
```

**Four keys you must understand:**
- `method` — the search algorithm. `grid` enumerates every combination, `random` samples i.i.d., `bayes` builds a Gaussian-process surrogate on `metric`.
- `metric` — only consumed by `bayes`. `grid` and `random` ignore it but still log it.
- `parameters` — a dict-of-dicts. Each inner dict specifies EITHER a distribution to sample from OR a discrete `values` list OR a fixed `value`.
- (optional) `name`, `program`, `early_terminate` — not required for a minimal sweep.

**You don't need the wandb package to BUILD the config.** It's just a Python dict — exercising the shape doesn't require any external dep.

### Exercise 1 — build a wandb sweep config: bayes + metric + parameters

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the wandb sweep config dict schema to specify a bayes-optimized minimization sweep with a mix of fixed, discrete, and continuous hyperparameters.
> Keywords: wandb, sweep, bayes, config-schema
> ```

**KCs targeted:** `sweep-config-top-level-schema`, `bayes-method-metric-block`

Implement `ex1_build_sweep_config(metric_name)`. Construct a valid wandb sweep config dict (you don't need wandb installed — this exercises the dict shape only).

The returned dict must contain:

1. `'method': 'bayes'` — bayesian optimization.
2. `'metric': {'name': metric_name, 'goal': 'minimize'}` — what to optimize.
3. `'parameters'`: a dict containing exactly these four entries:
   - `'lr'`: log-uniform distribution between `1e-5` and `1e-1`. Use `'log_uniform_values'`.
   - `'batch_size'`: discrete values `[16, 32, 64, 128]`.
   - `'optimizer'`: fixed at `'adam'`. Use `'value'`, not `'values'`.
   - `'weight_decay'`: log-uniform between `1e-6` and `1e-2`.

No wandb import needed; we exercise the dict shape only.

Output: `dict` matching the wandb sweep schema.

In [ ]:
def ex1_build_sweep_config(metric_name):
    return {
        'method': 'bayes',
        'metric': {
            'name': metric_name,
            'goal': 'minimize',
        },
        'parameters': {
            'lr': {
                'distribution': 'log_uniform_values',
                'min': 1e-5,
                'max': 1e-1,
            },
            'batch_size': {'values': [16, 32, 64, 128]},
            'optimizer': {'value': 'adam'},
            'weight_decay': {
                'distribution': 'log_uniform_values',
                'min': 1e-6,
                'max': 1e-2,
            },
        },
    }


<details><summary>Solution</summary>

```python
def ex1_build_sweep_config(metric_name):
    return {
        'method': 'bayes',
        'metric': {
            'name': metric_name,
            'goal': 'minimize',
        },
        'parameters': {
            'lr': {
                'distribution': 'log_uniform_values',
                'min': 1e-5,
                'max': 1e-1,
            },
            'batch_size': {'values': [16, 32, 64, 128]},
            'optimizer': {'value': 'adam'},
            'weight_decay': {
                'distribution': 'log_uniform_values',
                'min': 1e-6,
                'max': 1e-2,
            },
        },
    }
```

**`value` vs `values` is the #1 sweep-config typo.** Singular `value` = fixed (don't sweep). Plural `values` = discrete set (do sweep, sample uniformly). Get this wrong and wandb either runs your sweep over a single point (`values: 'adam'` parses as a 4-character list...) or treats your discrete set as a fixed string.

**Why `log_uniform_values` not `log_uniform`.** wandb has two log-uniform distributions: `log_uniform` (min/max in log-space — confusing) and `log_uniform_values` (min/max in linear value space, log-uniformly sampled — what you actually want). The `_values` suffix means 'specify the bounds as actual values, not as logs'.

**JSON-serializability is a hard requirement.** wandb ships the sweep config to its server; numpy floats, tensors, dataclasses, etc. all break it. Stick to ints, floats, strs, lists, dicts.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()